# LangChain 
## RAG 
This notebook covers:
1. RAG Without LangChain (manual implementation)
2. RAG With LangChain (interactive component visualization)


Model used throughout: `gpt-4o-mini`

---
## SECTION 0 - Installation and Setup

In [12]:
import subprocess, sys

packages = [
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-community",
    "langchain-core",
    "langgraph",
    "langsmith",
    "faiss-cpu",
    "tiktoken",
    "numpy",
    "requests",
    "python-dotenv",
    "chromadb"   
]

print("📦 Installing dependencies...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

print("✅ All dependencies installed successfully")

📦 Installing dependencies...
✅ All dependencies installed successfully



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import os

# Set your API keys here directly or via environment
# Replace the placeholder strings with your actual keys

os.environ["OPENAI_API_KEY"] = "sk-..."          # <-- replace with your OpenAI key
os.environ["LANGCHAIN_API_KEY"] = ""      # <-- replace with your LangSmith key
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "langchain-complete-guide"

# Optional: weather API key for real weather data (we will mock if not set)
os.environ["OPENWEATHER_API_KEY"] = ""             # <-- optional, leave blank to use mock

print("[SETUP] Environment variables configured.")
print(f"[SETUP] OpenAI key set: {'Yes' if os.environ.get('OPENAI_API_KEY', '').startswith('sk-') else 'No - please update'}")
print(f"[SETUP] LangSmith key set: {'Yes' if os.environ.get('LANGCHAIN_API_KEY', '').startswith('lsv2') else 'No - tracing will be disabled'}")

[SETUP] Environment variables configured.
[SETUP] OpenAI key set: Yes
[SETUP] LangSmith key set: Yes


In [56]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # won't overwrite an already-set env var

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f'✅ API key loaded: sk-...{api_key[-4:]}')
else:
    print('⚠️  OPENAI_API_KEY not found.')
    print('   Create a .env file in this folder with: OPENAI_API_KEY=sk-...')

from openai import OpenAI
client = OpenAI()  # automatically reads OPENAI_API_KEY from environment
print('\n✅ OpenAI client initialized')


✅ API key loaded: sk-...No0A

✅ OpenAI client initialized


---
## SECTION 1 - RAG Without LangChain

We build every component manually:
- Chunking
- Embedding
- Vector store (in-memory, using cosine similarity)
- Retrieval
- Generation

In [5]:
# ---------------------------------------------------------------
# COMPONENT 1: Document (raw text we want to query against)
# ---------------------------------------------------------------

RAW_DOCUMENT = """
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.

LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.

LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.
LangSmith integrates seamlessly with LangChain and LangGraph.

RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge base and provided to the LLM as context before generating a response.
RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.

Embeddings are numerical vector representations of text. Similar texts produce similar embedding vectors.
This property allows us to find semantically similar documents using distance metrics like cosine similarity.
OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models.

Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.
Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant.

Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, such as a web search, calculator, or API call.
Agents decide which tools to use and in what order based on the user's query.
"""

print("[RAW DOCUMENT LOADED]")
print(f"Total characters: {len(RAW_DOCUMENT)}")
print(f"Total words: {len(RAW_DOCUMENT.split())}")

[RAW DOCUMENT LOADED]
Total characters: 1948
Total words: 278


In [53]:
from langchain_community.document_loaders import PyPDFLoader

# Load RAW_DOCUMENT from a PDF file (overwrites current RAW_DOCUMENT)

pdf_path = "hrpolicy.pdf"  # update this path


loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

RAW_DOCUMENT = "\n\n".join(doc.page_content.strip() for doc in pdf_docs if doc.page_content.strip())

print(f"[PDF LOAD] Loaded {len(pdf_docs)} pages from: {pdf_path}")
print(f"[PDF LOAD] RAW_DOCUMENT chars: {len(RAW_DOCUMENT)}")
print(f"[PDF LOAD] RAW_DOCUMENT words: {len(RAW_DOCUMENT.split())}")
print("\n[PDF LOAD] Preview:\n")
print(RAW_DOCUMENT[:800])

[PDF LOAD] Loaded 209 pages from: hrpolicy.pdf
[PDF LOAD] RAW_DOCUMENT chars: 401699
[PDF LOAD] RAW_DOCUMENT words: 60643

[PDF LOAD] Preview:

HUMAN RESOURCES
POLICY MANUAL 2025
STAFF
www.iima.ac.in

i
IIMA HR Policy Manual 2025
DECLARATION
The objective of this Manual is to compile  the HR policies and 
procedures followed in IIMA. It also presents the general rules and 
regulations  that govern the employees of the Institute. 
This Manual supersedes all previous manuals, handbooks, and 
memorandums that may have been issued from time to time on 
subjects covered in this Manual.
The Institute reserves its right to interpret; change; suspend; cancel; 
or dispute, with or without notice; all or any part of what is contained 
in the Manual. The Institute will notify all employees of such changes.  
In the interpretation of any policies and procedures covered in 
the Manual, the Director’s decision will be final and binding on all 



In [54]:
# ---------------------------------------------------------------
# COMPONENT 2: Chunking
# Split the document into overlapping chunks
# ---------------------------------------------------------------

def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list:
    """
    Split text into overlapping character-level chunks.
    
    Args:
        text: The raw text to chunk.
        chunk_size: Max characters per chunk.
        overlap: Characters shared between consecutive chunks.
    
    Returns:
        List of chunk strings.
    """
    # Strip leading/trailing whitespace from paragraphs first
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    
    chunks = []
    for para in paragraphs:
        start = 0
        while start < len(para):
            end = start + chunk_size
            chunk = para[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += chunk_size - overlap
    
    return chunks


chunks = chunk_text(RAW_DOCUMENT, chunk_size=250, overlap=50)

print(f"[CHUNKING] Produced {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (len={len(chunk)}) ---")
    print(chunk)
    print()

[CHUNKING] Produced 2114 chunks

--- Chunk 1 (len=55) ---
HUMAN RESOURCES
POLICY MANUAL 2025
STAFF
www.iima.ac.in

--- Chunk 2 (len=250) ---
i
IIMA HR Policy Manual 2025
DECLARATION
The objective of this Manual is to compile  the HR policies and 
procedures followed in IIMA. It also presents the general rules and 
regulations  that govern the employees of the Institute. 
This Manual super

--- Chunk 3 (len=250) ---
the employees of the Institute. 
This Manual supersedes all previous manuals, handbooks, and 
memorandums that may have been issued from time to time on 
subjects covered in this Manual.
The Institute reserves its right to interpret; change; suspend;

--- Chunk 4 (len=248) ---
reserves its right to interpret; change; suspend; cancel; 
or dispute, with or without notice; all or any part of what is contained 
in the Manual. The Institute will notify all employees of such changes.  
In the interpretation of any policies and

--- Chunk 5 (len=185) ---
nges.  
In the interpretati

In [57]:
# ---------------------------------------------------------------
# COMPONENT 3: Embeddings
# Use OpenAI embeddings API directly (no LangChain)
# ---------------------------------------------------------------
import os
from openai import OpenAI
import numpy as np

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def get_embeddings(texts: list, model: str = "text-embedding-3-small") -> list:
    """
    Call OpenAI Embeddings API and return list of float vectors.
    """
    print(f"[EMBEDDINGS] Requesting embeddings for {len(texts)} texts using model: {model}")
    response = client.embeddings.create(input=texts, model=model)
    vectors = [item.embedding for item in response.data]
    print(f"[EMBEDDINGS] Received {len(vectors)} vectors, each of dimension {len(vectors[0])}")
    return vectors


# Embed all chunks
chunk_embeddings = get_embeddings(chunks)

# Show a preview of the first embedding
print(f"\n[EMBEDDINGS] Preview of Chunk 1 embedding (first 8 dimensions):")
print(np.round(chunk_embeddings[0][:8], 5))

[EMBEDDINGS] Requesting embeddings for 2114 texts using model: text-embedding-3-small


BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'input': array length must be 2048 or less.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [8]:
# ---------------------------------------------------------------
# COMPONENT 4: In-Memory Vector Store
# Store chunks + embeddings, support cosine similarity search
# ---------------------------------------------------------------

class SimpleVectorStore:
    """
    Minimal in-memory vector store using cosine similarity.
    No external libraries required beyond numpy.
    """

    def __init__(self):
        self.texts = []
        self.embeddings = []  # list of numpy arrays

    def add(self, texts: list, embeddings: list):
        self.texts.extend(texts)
        self.embeddings.extend([np.array(e) for e in embeddings])
        print(f"[VECTOR STORE] Added {len(texts)} documents. Total: {len(self.texts)}")

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def search(self, query_embedding: list, top_k: int = 3) -> list:
        """
        Return top_k most similar chunks and their similarity scores.
        """
        q = np.array(query_embedding)
        scores = [(self.cosine_similarity(q, e), t) for e, t in zip(self.embeddings, self.texts)]
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:top_k]


# Build the store
vector_store = SimpleVectorStore()
vector_store.add(chunks, chunk_embeddings)
print(f"[VECTOR STORE] Ready. Holding {len(vector_store.texts)} chunks.")

[VECTOR STORE] Added 14 documents. Total: 14
[VECTOR STORE] Ready. Holding 14 chunks.


In [ ]:
# ---------------------------------------------------------------
# COMPONENT 5: Retrieval + Generation (Full RAG Pipeline)
# ---------------------------------------------------------------

def rag_query_no_langchain(question: str, top_k: int = 3) -> str:
    """
    Full RAG pipeline without any LangChain dependency.
    Steps:
      1. Embed the question
      2. Retrieve top_k similar chunks
      3. Build a prompt with retrieved context
      4. Call GPT-4o-mini to generate the answer
    """
    print(f"\n{'='*60}")
    print(f"[RAG QUERY] Question: {question}")
    print(f"{'='*60}")

    # Step 1: Embed the query
    print("\n[STEP 1] Embedding the question...")
    query_embedding = get_embeddings([question])[0]

    # Step 2: Retrieve
    print(f"\n[STEP 2] Retrieving top {top_k} chunks from vector store...")
    results = vector_store.search(query_embedding, top_k=top_k)
    for rank, (score, text) in enumerate(results, 1):
        print(f"  Rank {rank} | Similarity: {score:.4f}")
        print(f"  Text: {text[:120]}...")
        print()

    # Step 3: Build context
    context = "\n\n".join([text for _, text in results])
    prompt = f"""You are a helpful assistant. Use ONLY the context below to answer the question.
If the context does not contain the answer, say "I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer:"""

    print(f"[STEP 3] Prompt built. Context length: {len(context)} chars.")

    # Step 4: Generate
    print("\n[STEP 4] Calling GPT-4o-mini for answer generation...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a concise and accurate assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content
    print(f"\n[ANSWER]\n{answer}")
    return answer


# Test it
answer1 = rag_query_no_langchain("What is RAG and why is it useful?")


[RAG QUERY] Question: What is RAG and why is it useful?

[STEP 1] Embedding the question...
[EMBEDDINGS] Requesting embeddings for 1 texts using model: text-embedding-3-small
[EMBEDDINGS] Received 1 vectors, each of dimension 1536

[STEP 2] Retrieving top 3 chunks from vector store...
  Rank 1 | Similarity: 0.6538
  Text: RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge...

  Rank 2 | Similarity: 0.2840
  Text: s represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and b...

  Rank 3 | Similarity: 0.2717
  Text: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provid...

[STEP 3] Prompt built. Context length: 617 chars.

[STEP 4] Calling GPT-4o-mini for answer generation...

[ANSWER]
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from 

In [10]:
# Test with another question
answer2 = rag_query_no_langchain("What is LangGraph and how does it differ from LangChain?")


[RAG QUERY] Question: What is LangGraph and how does it differ from LangChain?

[STEP 1] Embedding the question...
[EMBEDDINGS] Requesting embeddings for 1 texts using model: text-embedding-3-small
[EMBEDDINGS] Received 1 vectors, each of dimension 1536

[STEP 2] Retrieving top 3 chunks from vector store...
  Rank 1 | Similarity: 0.7406
  Text: LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It ...

  Rank 2 | Similarity: 0.6884
  Text: run.
LangSmith integrates seamlessly with LangChain and LangGraph....

  Rank 3 | Similarity: 0.5696
  Text: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provid...

[STEP 3] Prompt built. Context length: 548 chars.

[STEP 4] Calling GPT-4o-mini for answer generation...

[ANSWER]
LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs using a graph-based arch

---
## SECTION 2 - RAG With LangChain

Same pipeline but using LangChain components. We log each component explicitly so the internals are visible.

In [58]:
# ---------------------------------------------------------------
# COMPONENT 1: Document Loading and Chunking with LangChain
# ---------------------------------------------------------------

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Wrap raw text in a LangChain Document object
raw_doc = Document(page_content=RAW_DOCUMENT, metadata={"source": "manual", "title": "LangChain Overview"})

print("[DOCUMENT LOADING]")
print(f"Document metadata: {raw_doc.metadata}")
print(f"Content length: {len(raw_doc.page_content)} chars")

# LangChain text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

lc_chunks = splitter.split_documents([raw_doc])

print(f"\n[CHUNKING] RecursiveCharacterTextSplitter produced {len(lc_chunks)} chunks")
for i, chunk in enumerate(lc_chunks):
    print(f"--- Chunk {i+1} (len={len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print(f"Metadata: {chunk.metadata}")
    print()

[DOCUMENT LOADING]
Document metadata: {'source': 'manual', 'title': 'LangChain Overview'}
Content length: 401699 chars

[CHUNKING] RecursiveCharacterTextSplitter produced 2076 chunks
--- Chunk 1 (len=55) ---
HUMAN RESOURCES
POLICY MANUAL 2025
STAFF
www.iima.ac.in
Metadata: {'source': 'manual', 'title': 'LangChain Overview'}

--- Chunk 2 (len=231) ---
i
IIMA HR Policy Manual 2025
DECLARATION
The objective of this Manual is to compile  the HR policies and 
procedures followed in IIMA. It also presents the general rules and 
regulations  that govern the employees of the Institute.
Metadata: {'source': 'manual', 'title': 'LangChain Overview'}

--- Chunk 3 (len=225) ---
This Manual supersedes all previous manuals, handbooks, and 
memorandums that may have been issued from time to time on 
subjects covered in this Manual.
The Institute reserves its right to interpret; change; suspend; cancel;
Metadata: {'source': 'manual', 'title': 'LangChain Overview'}

--- Chunk 4 (len=211) ---
or dispute,

In [60]:
# ---------------------------------------------------------------
# COMPONENT 2: Embeddings with LangChain
# ---------------------------------------------------------------

import os
from langchain_openai import OpenAIEmbeddings

print("[EMBEDDINGS] Initializing LangChain OpenAIEmbeddings...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

# Embed a single test text to show the vector
test_text = "What is retrieval augmented generation?"
test_vector = embeddings_model.embed_query(test_text)

print(f"[EMBEDDINGS] Model: text-embedding-3-small")
print(f"[EMBEDDINGS] Embedding dimension: {len(test_vector)}")
print(f"[EMBEDDINGS] Query: '{test_text}'")
print(f"[EMBEDDINGS] First 8 dimensions: {[round(v, 5) for v in test_vector[:8]]}")

[EMBEDDINGS] Initializing LangChain OpenAIEmbeddings...
[EMBEDDINGS] Model: text-embedding-3-small
[EMBEDDINGS] Embedding dimension: 1536
[EMBEDDINGS] Query: 'What is retrieval augmented generation?'
[EMBEDDINGS] First 8 dimensions: [0.02075, -0.00279, 0.04276, 0.01639, -0.02719, -0.03091, 0.02672, 0.02127]


In [4]:
import subprocess, sys

print("📦 Installing ChromaDB...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "chromadb"],
    check=True
)

print("✅ ChromaDB installed successfully")

📦 Installing ChromaDB...
✅ ChromaDB installed successfully



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [61]:
# ---------------------------------------------------------------
# COMPONENT 3: Vector Store with ChromaDB
# ---------------------------------------------------------------

from langchain_community.vectorstores import Chroma

print("[VECTOR STORE] Building Chroma vector store from chunks...")

chroma_store = Chroma.from_documents(
    documents=lc_chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_db"  # optional (for saving DB)
)

print("[VECTOR STORE] Chroma DB created.")
print(f"[VECTOR STORE] Number of indexed documents: {len(lc_chunks)}")

# Test similarity search directly
print("\n[VECTOR STORE] Test similarity search for 'vector stores and FAISS':")

test_results = chroma_store.similarity_search_with_score(
    "vector stores and FAISS", 
    k=3
)

for rank, (doc, score) in enumerate(test_results, 1):
    print(f"  Rank {rank} | Score: {score:.4f}")
    print(f"  Text: {doc.page_content[:100]}...")
    print()

# Persistence is automatic in ChromaDB 0.4.x+

[VECTOR STORE] Building Chroma vector store from chunks...
[VECTOR STORE] Chroma DB created.
[VECTOR STORE] Number of indexed documents: 2076

[VECTOR STORE] Test similarity search for 'vector stores and FAISS':
  Rank 1 | Score: 0.5924
  Text: Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook A...

  Rank 2 | Score: 0.5924
  Text: Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook A...

  Rank 3 | Score: 0.9508
  Text: Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant....



In [62]:
# ---------------------------------------------------------------
# COMPONENT 4: Retriever
# ---------------------------------------------------------------

retriever = chroma_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("[RETRIEVER] Created Chroma retriever.")
print(f"[RETRIEVER] Search type: similarity | Top-k: 3")

# Test retriever
test_query = "How do agents use tools?"
retrieved_docs = retriever.invoke(test_query)
print(f"\n[RETRIEVER] Test query: '{test_query}'")
print(f"[RETRIEVER] Retrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  Doc {i}: {doc.page_content[:120]}...")

[RETRIEVER] Created Chroma retriever.
[RETRIEVER] Search type: similarity | Top-k: 3

[RETRIEVER] Test query: 'How do agents use tools?'
[RETRIEVER] Retrieved 3 documents:
  Doc 1: Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, suc...
  Doc 2: Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, suc...
  Doc 3: LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic....


In [63]:
# ---------------------------------------------------------------
# COMPONENT 5: Prompt Template + LLM + RAG Chain
# ---------------------------------------------------------------

import os
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# LLM
print("[LLM] Initializing ChatOpenAI with gpt-4o-mini...")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

# Prompt template
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain the answer, respond with: "I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer:
""")

print("[PROMPT] RAG prompt template created.")

# Helper to format retrieved documents
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Build the chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[CHAIN] RAG chain assembled using LCEL:")
print("  Retriever -> format_docs -> Prompt -> LLM -> StrOutputParser")

[LLM] Initializing ChatOpenAI with gpt-4o-mini...
[PROMPT] RAG prompt template created.
[CHAIN] RAG chain assembled using LCEL:
  Retriever -> format_docs -> Prompt -> LLM -> StrOutputParser


In [64]:
# ---------------------------------------------------------------
# Run RAG Queries with LangChain
# ---------------------------------------------------------------

def rag_query_langchain(question: str):
    print(f"\n{'='*60}")
    print(f"[RAG QUERY - LangChain] {question}")
    print(f"{'='*60}")
    
    # Show what is retrieved before the LLM sees it
    print("\n[RETRIEVAL LOG] Fetching relevant chunks...")
    retrieved = retriever.invoke(question)
    for i, doc in enumerate(retrieved, 1):
        print(f"  Chunk {i}: {doc.page_content[:120]}...")
    
    print("\n[LLM] Generating answer...")
    answer = rag_chain.invoke(question)
    print(f"\n[ANSWER]\n{answer}")
    return answer


# Run queries
rag_query_langchain("What is LangSmith used for?")


[RAG QUERY - LangChain] What is LangSmith used for?

[RETRIEVAL LOG] Fetching relevant chunks...
  Chunk 1: LangSmith integrates seamlessly with LangChain and LangGraph....
  Chunk 2: LangSmith integrates seamlessly with LangChain and LangGraph....
  Chunk 3: LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides traci...

[LLM] Generating answer...

[ANSWER]
LangSmith is used for debugging, testing, evaluating, and monitoring LLM applications.


'LangSmith is used for debugging, testing, evaluating, and monitoring LLM applications.'

In [67]:
rag_query_langchain("What is IIMA")


[RAG QUERY - LangChain] What is IIMA

[RETRIEVAL LOG] Fetching relevant chunks...
  Chunk 1: in 1961. IIMA has been conceived not only as a business school but also as a management institute. 
IIMA builds on over ...
  Chunk 2: rigour that matches the top league. With a distinguished faculty, an exceptional student-faculty 
ratio, and a 100-acre ...
  Chunk 3: and IIMA Regulations can be accessed on the IIMA Website....

[LLM] Generating answer...

[ANSWER]
IIMA is the Indian Institute of Management Ahmedabad, conceived as both a business school and a management institute, known for its excellence and leadership in management education.


'IIMA is the Indian Institute of Management Ahmedabad, conceived as both a business school and a management institute, known for its excellence and leadership in management education.'